In [ ]:
import dlt
import pyspark.sql.functions as F

# ==========================================
# 1. CONFORMED DIMENSIONS
# ==========================================

@dlt.table(
    name="gold.dim_store",
    comment="Grain: One row per store. Conformed dimension for row-level security and geography slicing."
)
def dim_store():
    return dlt.read("silver.silver_store")

@dlt.table(
    name="gold.dim_item",
    comment="Grain: One row per item. Conformed dimension for product category analytics."
)
def dim_item():
    return dlt.read("silver.silver_item")

@dlt.table(
    name="gold.dim_customer",
    comment="Grain: One row per customer. Conformed dimension for customer demographics and PII masking."
)
def dim_customer():
    return dlt.read("silver.silver_customer")

@dlt.table(
    name="gold.dim_reason",
    comment="Grain: One row per reason per effective time window (SCD Type 2). Conformed dimension for return root-cause analysis."
)
def dim_reason():
    return dlt.read("silver.silver_reason_history")

# ==========================================
# 2. CORE FACT TABLE
# ==========================================

# @dlt.table(
#     name="gold.fact_store_returns",
#     comment="Grain: One row per returned item per ticket. Core fact table capturing return metrics."
# )
# def fact_store_returns():
#     return dlt.read("silver.silver_store_returns").select(
#         F.col("ticket_number").alias("ticket_number", metadata={"comment": "Degenerate business key representing the transaction ticket"}),
#         F.col("item_sk").alias("item_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_item"}),
#         F.col("store_sk").alias("store_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_store (used for row-level security)"}),
#         F.col("customer_sk").alias("customer_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_customer"}),
#         F.col("reason_sk").alias("reason_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_reason"}),
#         F.col("return_quantity").alias("return_quantity", metadata={"comment": "Total units of item returned"}),
#         F.col("refunded_cash").alias("refunded_cash", metadata={"comment": "Total refund amount issued in cash"})
#     )

@dlt.table(
    name="gold.fact_store_returns",
    comment="Grain: One row per returned item per ticket. Core fact table capturing return metrics with row-filter dimension."
)
def fact_store_returns():
    returns_df = dlt.read("silver.silver_store_returns")
    store_df = dlt.read("silver.silver_store")
    
    return (
        returns_df.join(
            store_df,
            returns_df.store_sk == store_df.s_store_sk,
            "left"
        ).select(
            returns_df.returned_date_sk.alias("returned_date_sk", metadata={"comment": "Return date foreign key"}),
            returns_df.ticket_number.alias("ticket_number", metadata={"comment": "Degenerate business key representing the transaction ticket"}),
            returns_df.item_sk.alias("item_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_item"}),
            returns_df.store_sk.alias("store_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_store"}),
            store_df.s_state.alias("store_state", metadata={"comment": "Row-filter dimension: US state code where return occurred"}),
            returns_df.customer_sk.alias("customer_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_customer"}),
            returns_df.reason_sk.alias("reason_sk", metadata={"comment": "Surrogate foreign key referencing gold.dim_reason"}),
            returns_df.return_quantity.alias("return_quantity", metadata={"comment": "Total units of item returned"}),
            returns_df.refunded_cash.alias("refunded_cash", metadata={"comment": "Total refund amount issued in cash"})
        )
    )